In [30]:
# bibliotecas
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [33]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

transform = transforms.Compose([transforms.ToTensor()])

train_loader = DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=128, shuffle=True
)

In [38]:
# modelo VAE
class VAE(nn.Module):
    def __init__(self, latent_dim=2): # espaço latente
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder  = nn.Sequential(nn.Linear(784, 400), nn.ReLU())
        self.fc_mu = nn.Linear(400, latent_dim) # media da distribuiçao
        self.fc_logvar= nn.Linear(400, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 400), nn.ReLU(),
            nn.Linear(400, 784), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar) # converte log
        eps = torch.randn_like(std)
        return mu + eps * std # amostra diferenciável do espaço latente

    def forward(self, x):
        h = self.encoder(x.view(-1, 784)) # achata e codifica
        mu, logvar = self.fc_mu(h), self.fc_logvar(h) # parametros da distribuição
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
# reconstrução regularizaçao
def loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

In [36]:
# treino
for epoch in range(10):
    total_loss = 0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        recon, mu, logvar = model(imgs)
        loss = loss_function(recon, imgs, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss        += loss.item()
    print(f"epoca {epoch+1} loss: {total_loss / len(train_loader.dataset):.2f}")

epoca 1 loss: 190.45
epoca 2 loss: 168.29
epoca 3 loss: 163.69
epoca 4 loss: 161.18
epoca 5 loss: 159.21
epoca 6 loss: 157.75
epoca 7 loss: 156.59
epoca 8 loss: 155.54
epoca 9 loss: 154.65
epoca 10 loss: 153.87
